# Emerging Technologies Problems Notebook

This assignment explores the Deutsch–Jozsa algorithm, one of the earliest and most instructive examples of a quantum algorithm that outperforms any classical equivalent. The problem is deceptively simple: given a Boolean function that is guaranteed to be either constant or balanced, determine which one it is. Classically, answering this with certainty requires evaluating the function on just over half of all possible inputs — an exponential number of calls as the input size grows. The Deutsch–Jozsa algorithm answers the same question with a single query, regardless of how many input bits the function takes.

The assignment builds towards this result incrementally across five problems. Problem 1 establishes the mathematical setting by implementing a generator for random constant and balanced functions over four Boolean inputs — the functions the algorithm will ultimately be tested on. Problem 2 then tackles the problem classically, making the computational cost of the brute-force approach concrete and giving a baseline to compare against. Problems 3 and 4 introduce the quantum machinery, starting with the single-input case: Problem 3 constructs the four quantum oracles needed to represent every possible single-bit Boolean function, and Problem 4 implements Deutsch's algorithm — the simplest instance of the general approach — demonstrating how superposition and interference can extract a global property of a function from a single evaluation. Problem 5 then brings everything together, generalising the circuit to four input qubits and running it against the functions generated in Problem 1, with the classical solver from Problem 2 used to verify the quantum result at each step.

In [38]:
# imports used in this notebook

import random
from itertools import product
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import AerSimulator
import pytest

## Problem 1: Generating Random Boolean Functions

**Brief:** The Deutsch–Jozsa algorithm is designed to work with functions that accept a fixed number of Boolean inputs and return a single Boolean output. Each function is guaranteed to be either constant (always returns False or always returns True) or balanced (returns True for exactly half of the possible input combinations). Write a Python function random_constant_balanced that returns a randomly chosen function from the set of constant or balanced functions taking four Boolean arguments as inputs.

**random_constant_balanced() function:**

The random_constant_balanced function generates and returns a random Boolean function of four inputs (a, b, c, d) that satisfies the Deutsch–Jozsa promise: the function is guaranteed to be either constant or balanced. It first randomly decides whether to create a constant or balanced function. If constant, it randomly selects True or False and returns a function that ignores its inputs and always outputs that value. If balanced, it generates all 16 possible input combinations for four Boolean variables and randomly selects exactly 8 of them to return True, with the remaining 8 returning False. The returned function then checks whether its input tuple belongs to the selected set.

**Functioning:**

The random_constant_balanced function generates and returns a random Boolean function of four inputs (a, b, c, d) that satisfies the Deutsch–Jozsa promise: every function it produces is guaranteed to be either constant or balanced — never somewhere in between.

* The function begins by making a random binary decision using random.choice([True, False]). This gives an equal 50% probability of producing a constant function versus a balanced one. The result is stored in is_constant, which acts as a branch condition for the two possible construction paths below.

* If is_constant is True, the function takes the constant path. It randomly selects either True or False as the fixed output value. It then defines an inner function constant_function that accepts the four Boolean arguments (a, b, c, d) but completely ignores them — no matter what inputs are provided, it always returns the pre-selected value. This works because Python closures capture variables from their enclosing scope, so value is "baked in" at the time the inner function is created. The outer function then returns constant_function itself (not its result), so the caller receives a callable object.

* If is_constant is False, the function takes the balanced path. The first step is to enumerate the full domain of the function — all possible combinations of four Boolean values. product([False, True], repeat=4) is an iterator from Python's itertools module that generates the Cartesian product of {False, True} with itself four times. This yields all 24=162^4 = 16
24=16 ordered 4-tuples:
(False, False, False, False), (False, False, False, True), ..., (True, True, True, True). Wrapping it in list() materialises these into a concrete list so they can be randomly sampled in the next step.

* To ensure the function is balanced, exactly 8 of the 16 input tuples must be assigned the output True — and the remaining 8 must return False. random.sample(inputs, 8) selects 8 distinct tuples uniformly at random from the list of 16, without replacement. The result is then converted into a set, which is the ideal data structure here: membership testing with the in operator runs in O(1) average time for sets (versus O(n) for lists), making the returned function efficient to call regardless of how many times it is invoked.

* The inner function balanced_function takes the four Boolean arguments and packs them into the tuple (a, b, c, d). It then checks whether that tuple is a member of true_inputs — the randomly selected set of 8 "True" inputs — and returns the Boolean result of that membership test directly. Like constant_function above, balanced_function closes over true_inputs from the enclosing scope, so the set is fixed at creation time. The outer function returns balanced_function as a callable, completing the construction.

In [39]:
def random_constant_balanced():
    # Decide whether function is constant or balanced
    is_constant = random.choice([True, False])
    
    if is_constant:
        # Create a constant function
        value = random.choice([True, False])
        
        def constant_function(a, b, c, d):
            return value
        
        return constant_function
    
    else:        
        # Create a balanced function
        # Generate all 16 input combinations
        inputs = list(product([False, True], repeat=4))
        
        # Randomly choose 8 combinations to return True
        true_inputs = set(random.sample(inputs, 8))
        
        # check whether arguments in the list are in the selected 'True' set
        def balanced_function(a, b, c, d):
            return (a, b, c, d) in true_inputs
        
        return balanced_function

### **Problem 1 Testing**

In [40]:
def get_all_outputs(func):
    """
    Evaluates the function across all 16 possible Boolean input combinations
    and returns a list of its outputs. Since random_constant_balanced returns
    a *function*, we must call it exhaustively to observe its behaviour rather
    than inspecting its internals directly.
    """
    all_inputs = list(product([False, True], repeat=4))
    return [func(a, b, c, d) for (a, b, c, d) in all_inputs]

In [41]:
def classify(func):
    """
    Classifies a returned function as 'constant' or 'balanced' by counting
    how many of its 16 outputs are True. Returns the classification string,
    or raises if neither condition holds.
    """
    outputs = get_all_outputs(func)
    true_count = sum(outputs)
    if true_count == 0 or true_count == 16:
        return 'constant'
    elif true_count == 8:
        return 'balanced'
    else:
        raise AssertionError(f"Function is neither constant nor balanced — {true_count}/16 outputs are True.")

In [42]:
def print_truth_table(func):
    """
    Prints a formatted truth table for the function across all 16 inputs.
    This makes it visually clear which inputs map to True vs False, letting
    us directly see whether the function is constant or balanced.
    """
    all_inputs = list(product([False, True], repeat=4))
    print(f"  {'a':<6} {'b':<6} {'c':<6} {'d':<6} {'f(a,b,c,d)'}")
    print(f"  {'-'*42}")
    for (a, b, c, d) in all_inputs:
        result = func(a, b, c, d)
        # Mark True outputs with an arrow so they stand out visually
        marker = " <--" if result else ""
        print(f"  {str(a):<6} {str(b):<6} {str(c):<6} {str(d):<6} {str(result)}{marker}")


In [43]:
# ============================================================
# TEST 1: Return type is callable
# ============================================================

def test_returns_callable():
    """
    Verifies that random_constant_balanced() returns a callable function object,
    not a raw Boolean value or any other type. The DJ algorithm expects to receive
    an oracle it can query — not a pre-computed result.
    """
    print("\n" + "="*60)
    print("TEST 1: Return type is callable")
    print("="*60)
    print("Calling random_constant_balanced() and checking the return type...")

    result = random_constant_balanced()
    result_type = type(result).__name__

    print(f"  Return value: {result}")
    print(f"  Return type:  {result_type}")

    assert callable(result), (
        f"random_constant_balanced() should return a function, but got {result_type}."
    )

    print(f"  callable({result_type}): {callable(result)}")
    print("PASSED: The return value is a callable function object.")

In [44]:
# ============================================================
# TEST 2: Returned function accepts exactly four Boolean arguments
# ============================================================

def test_function_accepts_four_boolean_inputs():
    """
    Confirms the returned function can be called with four Boolean arguments
    without raising any errors. Four inputs is a hard requirement of the DJ
    problem — the function signature must be f(a, b, c, d).
    """
    print("\n" + "="*60)
    print("TEST 2: Returned function accepts four Boolean inputs")
    print("="*60)

    func = random_constant_balanced()
    print(f"  Generated function: {func.__name__}")
    print("\n  Calling with three representative input combinations:")

    test_cases = [
        (False, False, False, False),
        (True,  True,  True,  True),
        (True,  False, True,  False),
    ]

    for (a, b, c, d) in test_cases:
        output = func(a, b, c, d)
        print(f"    f({str(a):<5}, {str(b):<5}, {str(c):<5}, {str(d):<5}) = {output}")

    print("PASSED: Function accepted all four-argument calls without error.")

In [45]:
# ============================================================
# TEST 3: Output is always a Boolean
# ============================================================

def test_output_is_boolean():
    """
    Checks that the returned function always produces a strict Boolean (True or
    False) — not an integer, string, or other truthy/falsy value. The DJ oracle
    output feeds directly into quantum gate logic, so type correctness matters.
    """
    print("\n" + "="*60)
    print("TEST 3: Output is always a strict Boolean")
    print("="*60)

    func = random_constant_balanced()
    all_inputs = list(product([False, True], repeat=4))

    print(f"  Generated function: {func.__name__}")
    print(f"  Checking output type for all 16 inputs...\n")

    all_passed = True
    for (a, b, c, d) in all_inputs:
        output = func(a, b, c, d)
        is_bool = output is True or output is False
        status = "OK" if is_bool else "FAIL"

        # Only flag failures explicitly to keep output readable
        if not is_bool:
            print(f"    [{status}] f({a},{b},{c},{d}) = {output!r} (type: {type(output).__name__})")
            all_passed = False

    # Summarise at the end rather than printing every passing line
    true_count  = sum(func(a, b, c, d) for (a, b, c, d) in all_inputs)
    false_count = 16 - true_count
    print(f"  All 16 outputs checked:")
    print(f"    True  outputs: {true_count}")
    print(f"    False outputs: {false_count}")
    print(f"    All are strict booleans: {all_passed}")

    assert all_passed, "Some outputs were not strict Boolean values."
    print("PASSED: Every output is a strict Python Boolean.")

In [46]:
# ============================================================
# TEST 4: Every generated function satisfies the DJ promise
# ============================================================

def test_satisfies_dj_promise():
    """
    The core correctness test. Verifies that every function produced by
    random_constant_balanced is either perfectly constant or perfectly balanced.
    Repeated 200 times to stress-test both branches of the factory's logic.
    """
    print("\n" + "="*60)
    print("TEST 4: Every function satisfies the Deutsch-Jozsa promise")
    print("="*60)
    print("  Generating 200 functions and classifying each one...\n")

    results = {'constant': 0, 'balanced': 0, 'invalid': 0}
    invalid_examples = []

    for i in range(200):
        func = random_constant_balanced()
        outputs = get_all_outputs(func)
        true_count = sum(outputs)

        if true_count in (0, 16):
            results['constant'] += 1
        elif true_count == 8:
            results['balanced'] += 1
        else:
            results['invalid'] += 1
            invalid_examples.append((i, true_count))

    print(f"  Results across 200 generated functions:")
    print(f"    Constant (0 or 16 True outputs): {results['constant']}")
    print(f"    Balanced (exactly 8 True outputs): {results['balanced']}")
    print(f"    Invalid  (any other count):        {results['invalid']}")

    if invalid_examples:
        print(f"\n  Invalid cases detected:")
        for idx, count in invalid_examples:
            print(f"    Iteration {idx}: {count}/16 outputs were True")

    assert results['invalid'] == 0, (
        f"{results['invalid']} functions violated the DJ promise."
    )
    print("\nPASSED: All 200 functions were either constant or balanced.")


In [47]:
# ============================================================
# TEST 5: Constant functions return the same value for ALL inputs
# ============================================================

def test_constant_function_is_truly_constant():
    """
    When a constant function is produced, it must return the exact same value
    for every one of the 16 possible input combinations — not just most of them.
    We isolate a constant function and print its full truth table to demonstrate.
    """
    print("\n" + "="*60)
    print("TEST 5: Constant functions return the same value for all inputs")
    print("="*60)
    print("  Searching for a constant function to inspect...\n")

    constant_func = None
    attempts = 0
    for _ in range(1000):
        attempts += 1
        func = random_constant_balanced()
        outputs = get_all_outputs(func)
        if len(set(outputs)) == 1:
            constant_func = func
            constant_value = outputs[0]
            break

    assert constant_func is not None, "Could not generate a constant function in 1000 attempts."

    print(f"  Found a constant function after {attempts} attempt(s).")
    print(f"  Constant output value: {constant_value}")
    print(f"\n  Full truth table for this constant function:")
    print_truth_table(constant_func)

    outputs = get_all_outputs(constant_func)
    unique_outputs = set(outputs)
    print(f"\n  Unique output values seen across all 16 inputs: {unique_outputs}")

    assert len(unique_outputs) == 1, f"Expected 1 unique output, got: {unique_outputs}"
    print("PASSED: Constant function returned the same value for all 16 inputs.")



In [48]:
# ============================================================
# TEST 6: Constant functions cover both True and False variants
# ============================================================

def test_constant_functions_can_be_both_true_and_false():
    """
    Verifies the factory can produce both constant-True and constant-False
    functions. If only one variant were possible, the oracle set would be
    incomplete and biased.
    """
    print("\n" + "="*60)
    print("TEST 6: Constant functions can output both True and False")
    print("="*60)
    print("  Generating functions until both constant-True and constant-False are observed...\n")

    seen_constant_values = {}  # Maps True/False → attempt number first seen
    attempt = 0

    for _ in range(500):
        attempt += 1
        func = random_constant_balanced()
        outputs = get_all_outputs(func)

        if len(set(outputs)) == 1:
            val = outputs[0]
            if val not in seen_constant_values:
                seen_constant_values[val] = attempt
                print(f"  Found constant-{val} function on attempt {attempt}.")

        if seen_constant_values.keys() == {True, False}:
            break

    print(f"\n  Summary of constant variants observed:")
    for val, first_seen in sorted(seen_constant_values.items()):
        print(f"    constant-{str(val):<5}: first seen on attempt {first_seen}")

    assert True  in seen_constant_values, "Never generated a constant-True function."
    assert False in seen_constant_values, "Never generated a constant-False function."
    print("\nPASSED: Both constant-True and constant-False variants were produced.")

In [49]:
# ============================================================
# TEST 7: Balanced functions return True for exactly 8 of 16 inputs
# ============================================================

def test_balanced_function_has_exactly_eight_true_outputs():
    """
    When a balanced function is produced, exactly half (8 of 16) of its outputs
    must be True. We find a balanced function, print its truth table, and
    confirm the count is precisely 8.
    """
    print("\n" + "="*60)
    print("TEST 7: Balanced functions return True for exactly 8 of 16 inputs")
    print("="*60)
    print("  Searching for a balanced function to inspect...\n")

    balanced_func = None
    attempts = 0
    for _ in range(1000):
        attempts += 1
        func = random_constant_balanced()
        outputs = get_all_outputs(func)
        if sum(outputs) == 8:
            balanced_func = func
            break

    assert balanced_func is not None, "Could not generate a balanced function in 1000 attempts."

    outputs = get_all_outputs(balanced_func)
    true_count  = sum(outputs)
    false_count = 16 - true_count

    print(f"  Found a balanced function after {attempts} attempt(s).")
    print(f"\n  Full truth table for this balanced function:")
    print_truth_table(balanced_func)

    print(f"\n  Output count summary:")
    print(f"    True  outputs: {true_count}  (must be exactly 8)")
    print(f"    False outputs: {false_count}  (must be exactly 8)")

    assert true_count == 8, f"Expected 8 True outputs, got {true_count}."
    print("\nPASSED: Balanced function returned True for exactly 8 of 16 inputs.")

In [50]:
# ============================================================
# TEST 8: Balanced functions vary in which inputs map to True
# ============================================================

def test_balanced_functions_are_randomly_distributed():
    """
    Confirms the balanced branch produces genuine randomness — different calls
    should yield different assignments of which 8 inputs return True. We collect
    unique 'fingerprints' and display them to show the diversity.
    """
    print("\n" + "="*60)
    print("TEST 8: Balanced functions have randomly distributed True inputs")
    print("="*60)
    print("  Collecting unique True-input fingerprints from balanced functions...\n")

    all_inputs = list(product([False, True], repeat=4))
    fingerprints = {}  # frozenset → first attempt number seen

    attempt = 0
    for _ in range(300):
        attempt += 1
        func = random_constant_balanced()
        outputs = get_all_outputs(func)

        if sum(outputs) == 8:
            fp = frozenset(inp for inp, out in zip(all_inputs, outputs) if out)
            if fp not in fingerprints:
                fingerprints[fp] = attempt

        if len(fingerprints) >= 5:
            break

    print(f"  Found {len(fingerprints)} distinct balanced function(s) across {attempt} attempt(s).")
    print(f"\n  Sample of unique True-input sets discovered:")

    for i, (fp, first_seen) in enumerate(fingerprints.items(), 1):
        # Convert frozenset of tuples to a compact readable format
        sorted_inputs = sorted(fp)
        compact = [str(list(map(int, inp))) for inp in sorted_inputs]
        print(f"\n    Variant {i} (first seen attempt {first_seen}) — True for inputs:")
        # Print in rows of 4 for readability
        for j in range(0, len(compact), 4):
            print(f"      {'  '.join(compact[j:j+4])}")

    assert len(fingerprints) > 1, (
        "All balanced functions had identical True-input sets — randomness appears broken."
    )
    print(f"\nPASSED: {len(fingerprints)} distinct balanced functions observed — randomness confirmed.")



In [51]:
# ============================================================
# TEST 9: Both constant and balanced functions are produced
# ============================================================

def test_both_types_are_generated():
    """
    Checks that random_constant_balanced produces both constant and balanced
    functions across many calls. If only one type ever appeared, half the oracle
    space would be inaccessible.
    """
    print("\n" + "="*60)
    print("TEST 9: Both constant and balanced functions are generated")
    print("="*60)
    print("  Generating up to 200 functions and tracking which types appear...\n")

    type_counts = {'constant': 0, 'balanced': 0}
    first_seen  = {}
    attempt = 0

    for _ in range(200):
        attempt += 1
        func = random_constant_balanced()
        kind = classify(func)
        type_counts[kind] += 1

        if kind not in first_seen:
            first_seen[kind] = attempt
            print(f"  First '{kind}' function seen on attempt {attempt}.")

        if len(first_seen) == 2:
            break

    print(f"\n  Final counts after {attempt} attempt(s):")
    print(f"    Constant functions: {type_counts['constant']}")
    print(f"    Balanced functions: {type_counts['balanced']}")

    assert 'constant' in first_seen, "No constant function produced across 200 attempts."
    assert 'balanced' in first_seen, "No balanced function produced across 200 attempts."
    print("\nPASSED: Both constant and balanced functions were generated.")



In [52]:
# ============================================================
# TEST 10: Each call produces an independent function (no shared state)
# ============================================================

def test_each_call_is_independent():
    """
    Verifies that two functions generated by separate calls are fully independent
    closures — calling one must not affect the other. This guards against bugs
    where closures accidentally share mutable state.
    """
    print("\n" + "="*60)
    print("TEST 10: Each call produces an independent function (no shared state)")
    print("="*60)
    print("  Generating two functions from separate calls...\n")

    func1 = random_constant_balanced()
    func2 = random_constant_balanced()

    outputs1 = get_all_outputs(func1)
    outputs2 = get_all_outputs(func2)

    kind1 = classify(func1)
    kind2 = classify(func2)

    true_count1 = sum(outputs1)
    true_count2 = sum(outputs2)

    print(f"  func1: {func1.__name__:<20}  type={kind1:<10}  True outputs={true_count1}/16")
    print(f"  func2: {func2.__name__:<20}  type={kind2:<10}  True outputs={true_count2}/16")
    print(f"\n  Same object in memory? {func1 is func2}")
    print(f"  func1 id: {id(func1)}")
    print(f"  func2 id: {id(func2)}")

    # Check calling func2 exhaustively didn't corrupt func1's outputs
    outputs1_recheck = get_all_outputs(func1)
    print(f"\n  Re-checking func1 outputs after calling func2 exhaustively...")
    print(f"  Outputs match original: {outputs1 == outputs1_recheck}")

    assert func1 is not func2, "Both calls returned the same object — closures are not independent."
    assert outputs1_recheck == outputs1, "func1 outputs changed after calling func2 — shared state detected."
    assert true_count1 in (0, 8, 16), f"func1 violated the DJ promise: {true_count1}/16 True."
    assert true_count2 in (0, 8, 16), f"func2 violated the DJ promise: {true_count2}/16 True."
    print("\nPASSED: Both functions are independent objects with stable, isolated state.")


In [53]:
# ============================================================
# RUN ALL TESTS
# ============================================================

if __name__ == "__main__":
    tests = [
        test_returns_callable,
        test_function_accepts_four_boolean_inputs,
        test_output_is_boolean,
        test_satisfies_dj_promise,
        test_constant_function_is_truly_constant,
        test_constant_functions_can_be_both_true_and_false,
        test_balanced_function_has_exactly_eight_true_outputs,
        test_balanced_functions_are_randomly_distributed,
        test_both_types_are_generated,
        test_each_call_is_independent,
    ]

    passed = 0
    failed = 0
    failures = []

    for test in tests:
        try:
            test()
            passed += 1
        except AssertionError as e:
            failed += 1
            failures.append((test.__name__, str(e)))
            print(f"FAILED: {e}")

    print("\n" + "="*60)
    print("FINAL RESULTS")
    print("="*60)
    print(f"  Passed: {passed}/{len(tests)}")
    print(f"  Failed: {failed}/{len(tests)}")

    if failures:
        print("\n  Failed tests:")
        for name, reason in failures:
            print(f"    - {name}: {reason}")
    else:
        print("\n  All tests passed successfully.")


TEST 1: Return type is callable
Calling random_constant_balanced() and checking the return type...
  Return value: <function random_constant_balanced.<locals>.constant_function at 0x75769c14e520>
  Return type:  function
  callable(function): True
PASSED: The return value is a callable function object.

TEST 2: Returned function accepts four Boolean inputs
  Generated function: balanced_function

  Calling with three representative input combinations:
    f(False, False, False, False) = False
    f(True , True , True , True ) = True
    f(True , False, True , False) = True
PASSED: Function accepted all four-argument calls without error.

TEST 3: Output is always a strict Boolean
  Generated function: balanced_function
  Checking output type for all 16 inputs...

  All 16 outputs checked:
    True  outputs: 8
    False outputs: 8
    All are strict booleans: True
PASSED: Every output is a strict Python Boolean.

TEST 4: Every function satisfies the Deutsch-Jozsa promise
  Generating 20

In [54]:
f = random_constant_balanced()

print(f(False, False, False, False))
print(f(True, False, True, True))

True
False


## **Problem 2: Classical Testing for Function Type**

**Brief:** Deutsch's algorithm is designed to demonstrate a potential advantage of quantum computing over classical computation. To understand this advantage, we must first understand the classical cost of solving the underlying problem. Write a Python function determine_constant_balanced that takes as input a function f, as defined in Problem 1. The function should analyze f and return the string "constant" or "balanced" depending on whether the function is constant or balanced. Write a brief note on the efficiency of your solution. What is the maximum number of times you need to call f to be 100% certain whether it is constant or balanced?

### **determine_constant_balanced(f) Function:**

The function determine_constant_balanced determines whether a given 4-input Boolean function f is constant or balanced by evaluating the function on every possible input combination. Using product([False, True], repeat=4), the code generates all 16 possible combinations of four Boolean inputs. It then calls f on each combination and counts how many times the function returns True. 

If the count is either 0 or 16, all outputs are the same and the function is constant; otherwise, since the problem guarantees the function is either constant or balanced, it must be balanced (8 True and 8 False). The algorithm runs in n-input Boolean function because it checks every possible input. In the worst case, the function must call f 16 times to be completely certain of the result, since even after 15 identical outputs the final input could still determine whether the function is constant or balanced. This classical cost highlights the motivation behind Deutsch's algorithm, which can solve the problem with only one evaluation of the function on a quantum computer.

**Purpose:**

This function implements the classical approach to solving the problem that the Deutsch–Jozsa algorithm is designed to tackle quantumly. It takes as input a function f of the kind produced by random_constant_balanced() in Problem 1 — a 4-input Boolean function guaranteed to be either constant or balanced — and determines which of the two it is by exhaustively evaluating f across its entire input domain.

Because f is a black-box oracle (its internal structure is deliberately hidden, as it would be in a real quantum computing scenario), the only way to classify it classically is to call it repeatedly and observe its outputs. There is no shortcut: a classical algorithm cannot peek inside the function. It can only learn about f by querying it. This function therefore represents the brute-force classical baseline against which the quantum advantage of Deutsch's algorithm will be measured.


**Implementation:**

The function begins by constructing the complete input domain of f. Since f accepts four Boolean arguments, there are 24=162^4 = 16
24=16 possible input combinations. These are generated using
product([False, True], repeat=4) from Python's itertools module — the same utility used inside random_constant_balanced() to build the balanced function's truth set — which produces all ordered 4-tuples of False and True values. These are collected into a list called inputs.
Next, a counter true_count is initialised to zero. The function then iterates over every input combination in inputs, unpacking each 4-tuple into the four Boolean arguments a, b, c, d using Python's * unpacking operator, and calls f(*inp). Each time f returns True, true_count is incremented. By the end of the loop, true_count holds the total number of inputs for which f evaluated to True.
The classification then follows directly from the Deutsch–Jozsa promise established in Problem 1:

* If true_count is 0 or 16, every input produced the same output — the function is constant.
* Otherwise, since f is guaranteed to be either constant or balanced, the only remaining possibility is that exactly 8 inputs returned True and 8 returned False — the function is balanced.

No further checks are needed because the DJ promise rules out any other outcome. The function returns the appropriate string "constant" or "balanced".

* Step 1: Generate all 24=162^4 = 16 - 24=16 possible 4-tuple Boolean input combinations using product([False, True], repeat=4) and store them in a list.
* Step 2: Initialise a counter true_count to zero to track how many inputs cause f to return True.
* Step 3: Iterate over every input combination, calling f(*inp) for each, and increment true_count whenever f returns True.
* Step 4: After all 16 evaluations, check true_count: if it is 0 or 16, return "constant"; otherwise return "balanced".

**Efficiency:**

This solution runs in O(2^n) time for an n-input Boolean function, as it must evaluate f on every element of its input domain. For n=4, that means exactly 16 calls to f in all cases. Crucially, this exhaustive evaluation is not a pessimistic worst case that might sometimes be avoided — it is an unavoidable requirement of classical certainty. Even if the first 15 calls all return the same value the 16th call could still change the classification: 15 True outputs followed by one False would indicate a balanced function, while 16 True outputs would indicate a constant one. No classical algorithm can be 100% certain of the result without making that final call.
This is precisely the classical cost that motivates the Deutsch–Jozsa algorithm. Where a classical solver requires 2n2^n2n queries to
f to guarantee a correct answer, the quantum algorithm resolves the same question with a single query, achieving an exponential speedup. Problem 3 will implement that quantum approach and make this contrast concrete.


**Relation to other functions:**

determine_constant_balanced is the classical counterpart to the quantum circuit that will be built in the problems that follow. It consumes the oracle functions produced by random_constant_balanced() in Problem 1, and its output — the strings "constant" or "balanced" — provides a ground-truth reference that can be used to verify that the quantum implementation reaches the correct answer. Together, Problems 1 and 2 establish the full classical picture: how oracles are constructed, and what it costs to interrogate them without quantum resources.

In [55]:
from itertools import product 

def determine_constant_balanced(f):
    """
    Determines whether a given 4-input Boolean function f
    is constant or balanced.
    """

    # Generate all 16 possible input combinations for four Boolean variables
    inputs = list(product([False, True], repeat=4))

    # Counter to track how many times f returns True.
    true_count = 0

    # Evaluate f on every possible input
    for inp in inputs:
        if f(*inp):
            true_count += 1
            
    # If all outputs are the same → constant
    if true_count == 0 or true_count == 16:
        return "constant"
    else:
        # Since the problem guarantees constant or balanced,
        # the only remaining possibility is balanced (8 True, 8 False)
        return "balanced"

### **Problem 2 Testing**

In [ ]:
# ---- Test 1: Constant False ----
def constant_false(a, b, c, d):
    return False

assert determine_constant_balanced(constant_false) == "constant"
print("Test 1 passed: constant_false correctly identified.")


# ---- Test 2: Constant True ----
def constant_true(a, b, c, d):
    return True

assert determine_constant_balanced(constant_true) == "constant"
print("Test 2 passed: constant_true correctly identified.")


# ---- Test 3: Balanced Function (Parity Example) ----
# Returns True when an even number of inputs are True
def balanced_example(a, b, c, d):
    return (a + b + c + d) % 2 == 0

# This is balanced: exactly 8 inputs return True
assert determine_constant_balanced(balanced_example) == "balanced"
print("Test 3 passed: balanced_example correctly identified.")


# ---- Test 4: Randomly Generated Functions ----
# Tests compatibility with Problem 1 generator 
for i in range(5):
    f = random_constant_balanced()
    result = determine_constant_balanced(f)
    print(f"Random test {i+1}: Function classified as {result}")

print("Random tests executed successfully.")

Test 1 passed: constant_false correctly identified.
Test 2 passed: constant_true correctly identified.
Test 3 passed: balanced_example correctly identified.
Random test 1: Function classified as constant
Random test 2: Function classified as balanced
Random test 3: Function classified as constant
Random test 4: Function classified as constant
Random test 5: Function classified as balanced
Random tests executed successfully.


## **Problem 3: Quantum Oracles**

**Brief:** Deutsch's algorithm is the simplest example of a quantum algorithm using superposition to determine a global property of a function with a single evaluation. In the single-input case, there are four possible Boolean functions. Using Qiskit, create the appropriate quantum oracles for each of the possible single-Boolean-input functions used in Deutsch's algorithm. Demonstrate their use and explain how each oracle implements its corresponding function.

### **Introduction**

Problem 3 requires implementing quantum oracles for each of the four possible single-input Boolean functions. A quantum oracle is a black-box unitary operation that encodes a classical function f: {0,1} → {0,1} into a quantum circuit. Each oracle acts on two qubits using the transformation |x⟩|y⟩ → |x⟩|y ⊕ f(x)⟩, where x is the input qubit and y is the ancilla qubit. XOR-ing the ancilla with f(x) rather than overwriting it keeps the operation reversible, which is a requirement for all quantum gates. The four oracles implemented here will be used in Problem 4 to demonstrate Deutsch's algorithm, which determines whether a function is constant or balanced using only a single oracle call.

### **oracle_f0() - constant zero function:**

oracle_f0 implements the constant zero function f(x) = 0. Since f(x) = 0 for all x, the oracle transformation becomes |x⟩|y⟩ → |x⟩|y ⊕ 0⟩ = |x⟩|y⟩ — XOR-ing with 0 leaves y unchanged. As a result, neither qubit is modified and no gates are needed. The function body is simply a pass, which is itself a valid two-qubit identity unitary. This is the simplest possible oracle and serves as a useful baseline: any circuit that applies oracle_f0 is unchanged by it.

In [57]:
# f(x) = 0  (constant function)
def oracle_f0(qc, x, y):
    """
    f0(x) = 0  (constant zero)
    
    Oracle action: |x⟩|y⟩ → |x⟩|y XOR 0⟩ = |x⟩|y⟩
    
    XOR-ing with 0 changes nothing, so this oracle is the identity —
    no gates are needed at all. The ancilla qubit y is left untouched
    """
    
    # Does nothing because f(x) = 0 -> y XOR 0 = y
    return qc

### **oracle_f1(qc, x, y) - constant one function:**

The following function implements the constant one function f(x) = 1 -> since f(x) = 1 for all x, the oracle transformation becaomes  |x⟩|y⟩ → |x⟩|y ⊕ 1⟩ = |x⟩|NOT y⟩ — XOR-ing with 1 always flips the ancilla Q-bit not matter the value of x. This is emplemetned with a single Pauli-X gate on y, which is the quantum equivalent of a classical NOT gate (flips |0⟩ to |1⟩ and vice versa unconditionally). Unlike oracle_f2 and oracle_f3 (covered later in this problem), there is no dependency on x at all - the X gate is applied directly to y with no control Q-bit. This makes oracle_f1 the simplest non-trivial oracle: one gate, no conditioning, ancilla always flips. 

In [58]:
def oracle_f1(qc, x, y):
    """
    f1(x) = 1  (constant one)
    
    Oracle action: |x⟩|y⟩ → |x⟩|y XOR 1⟩ = |x⟩|NOT y⟩
    
    XOR-ing with 1 always flips y, regardless of x. A single X (Pauli-X /
    NOT) gate on the output qubit y implements this unconditional flip.
    """
    
    # f(x) = 1 for all x, so y XOR f(x) = y XOR 1 = NOT y.
    # X gate (Pauli-X) is the quantum NOT — flips |0⟩↔|1⟩ unconditionally.
    # No dependency on x: the flip happens regardless of the input qubit's state.
    qc.x(y)

### **oracle_f2(qc, x, y) - identity / balanced function:**

oracle_f2 implements the identity function f(x) = x. Since f(x) = x, the oracle transformation becomes |x⟩|y⟩ → |x⟩|y ⊕ x⟩ — the ancilla qubit y is flipped if and only if x = 1, and left unchanged if x = 0. This is implemented with a single CNOT gate, where x is the control qubit and y is the target. The CNOT is the natural quantum gate for this operation: it applies a Pauli-X to the target only when the control is |1⟩, which is exactly the behaviour y ⊕ x requires. Unlike oracle_f1 which flips y unconditionally, and oracle_f3 which flips y when x = 0, oracle_f2 flips y when x = 1 — making it the most direct mapping from a classical Boolean function to a quantum gate, as the function value and the control condition are one and the same.

In [59]:
def oracle_f2(qc, x, y):
    """
    f2(x) = x  (identity / balanced)
    
    Oracle action: |x⟩|y⟩ → |x⟩|y XOR x⟩
    
    XOR-ing y with x is exactly what a CNOT gate does: it flips y if and only
    if x = 1. A single CNOT with x as control and y as target implements this.
    """
    
    # f(x) = x, so y XOR f(x) = y XOR x — flip y iff x is |1⟩.
    # CNOT (controlled-X): x is the control qubit, y is the target.
    # |0⟩|y⟩ → |0⟩|y⟩  (x=0: no flip)
    # |1⟩|y⟩ → |1⟩|NOT y⟩  (x=1: flip y)
    qc.cx(x, y)

### **oracle_f3(qc, x, y) - negation / balanced function:**

oracle_f3 implements the negation function f(x) = NOT x. Since f(x) = NOT x, the oracle transformation becomes |x⟩|y⟩ → |x⟩|y ⊕ (NOT x)⟩ — the ancilla qubit y is flipped when x = 0, and left unchanged when x = 1. This is the opposite control condition to oracle_f2, and cannot be implemented with a standard CNOT alone since a CNOT fires when its control is |1⟩, not |0⟩. Instead, the implementation uses a three-gate sequence: an X gate is applied to x to temporarily invert it, turning the |0⟩ case into |1⟩ so the CNOT fires on the correct input, then a second X gate restores x to its original value. This flanking X-CNOT-X pattern is known as an anti-controlled NOT. The restore step in particular is essential — without it the oracle would modify the input qubit x, violating the unitarity requirement that |x⟩ must be left unchanged. This same pattern appears in scaled form in Problem 5's build_oracle, where X gates flank a multi-controlled X gate to condition it on arbitrary input patterns rather than just |0⟩.

In [60]:
def oracle_f3(qc, x, y):
    """
    f3(x) = NOT x  (negation / balanced)
    
    Oracle action: |x⟩|y⟩ → |x⟩|y XOR (NOT x)⟩
    
    NOT x = 1 XOR x, so:  y XOR (NOT x) = y XOR 1 XOR x
    
    We need to flip y when x = 0 (and leave it when x = 1). This is an
    anti-controlled NOT: X on y controlled by x being |0⟩. Implemented as:
        X(x) → CNOT(x,y) → X(x)
    The pair of X gates around the CNOT inverts the control logic.
    """
    # Step 1: Flip x so that the CNOT's control logic is inverted.
    # After X(x): |0⟩→|1⟩ and |1⟩→|0⟩, swapping which state triggers the CNOT.
    qc.x(x)

    # Step 2: CNOT fires when x was originally |0⟩ (now temporarily |1⟩).
    # This encodes the y XOR x part of: y XOR (NOT x) = y XOR 1 XOR x.
    qc.cx(x, y)

    # Step 3: Restore x to its original value — keeps the oracle's input qubit
    # unchanged, satisfying the unitary requirement |x⟩ → |x⟩.
    qc.x(x)


### **Problem 3 Testing**

#### Main function which automates tests

In [61]:
def run_oracle(oracle_fn, x_init: int, y_init: int) -> tuple[int, int]:
    """
    Build a two-qubit circuit, initialise |x⟩|y⟩ to the given basis state,
    apply the oracle, measure both qubits, and return (x_measured, y_measured).
 
    Because the inputs are computational basis states (no superposition), the
    result is deterministic — 1024 shots all produce the same bitstring.
 
    Qiskit's measurement string is ordered q1q0 (most-significant bit first),
    so we reverse it before extracting individual qubit values.
    """
    qr = QuantumRegister(2, 'q')
    cr = ClassicalRegister(2, 'c')
    qc = QuantumCircuit(qr, cr)
 
    # Initialise input qubit x (qr[0]) to |x_init⟩
    if x_init == 1:
        qc.x(qr[0])
 
    # Initialise ancilla qubit y (qr[1]) to |y_init⟩
    if y_init == 1:
        qc.x(qr[1])
 
    # Apply the oracle under test
    oracle_fn(qc, qr[0], qr[1])
 
    # Measure both qubits
    qc.measure(qr, cr)
 
    # Simulate and extract the single deterministic outcome
    simulator = AerSimulator()
    result = simulator.run(qc, shots=1024).result()
    counts = result.get_counts()
 
    # counts keys are bitstrings ordered "c1c0"; reverse to get [c0, c1]
    bitstring = max(counts, key=counts.get)[::-1]
    x_out = int(bitstring[0])   # qubit 0 — the input qubit x
    y_out = int(bitstring[1])   # qubit 1 — the ancilla qubit y
 
    return x_out, y_out

#### **oracle_f0 Function Tests:**

In [62]:
# Tests oracle_f0 with 00 input 
def test_f0_input_00():
    # f0 is the constant zero function, so f0(0) = 0.
    # The oracle computes y XOR f0(x) = 0 XOR 0 = 0, so y is unchanged.
    # Since f0 never depends on x, x is also unchanged — both qubits stay 0.
    x_out, y_out = run_oracle(oracle_f0, x_init=0, y_init=0)
    assert x_out == 0 and y_out == 0  # |0⟩|0⟩ → |0⟩|0⟩  (0 XOR 0 = 0)
    print(f"f0 |0⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f0 with 01 input 
def test_f0_input_01():
    # f0 is the constant zero function, so f0(1) = 0.
    # The oracle computes y XOR f0(x) = 0 XOR 0 = 0, so y is unchanged.
    # This confirms that even when x = 1, f0 still returns 0 — it is constant.
    x_out, y_out = run_oracle(oracle_f0, x_init=0, y_init=1)
    assert x_out == 0 and y_out == 1  # |0⟩|1⟩ → |0⟩|1⟩  (1 XOR 0 = 1)
    print(f"f0 |0⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f0 with 10 input 
def test_f0_input_10():
    # f0 is the constant zero function, so f0(1) = 0.
    # The oracle computes y XOR f0(x) = 0 XOR 0 = 0, so y is unchanged.
    # This confirms that even when x = 1, f0 still returns 0 — it is constant.
    x_out, y_out = run_oracle(oracle_f0, x_init=1, y_init=0)
    assert x_out == 1 and y_out == 0  # |1⟩|0⟩ → |1⟩|0⟩  (0 XOR 0 = 0)
    print(f"f0 |1⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f0 with 11 input 
def test_f0_input_11():
    # f0 is the constant zero function, so f0(1) = 0.
    # The oracle computes y XOR f0(x) = 1 XOR 0 = 1, so y remains 1.
    # Together, all four tests confirm that oracle_f0 leaves every input
    # unchanged — consistent with XOR-ing by 0 being the identity operation.
    x_out, y_out = run_oracle(oracle_f0, x_init=1, y_init=1)
    assert x_out == 1 and y_out == 1  # |1⟩|1⟩ → |1⟩|1⟩  (1 XOR 0 = 1)
    print(f"f0 |1⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")

#### **oracle_f1 Function Tests:**

In [63]:
# Tests oracle_f1 with 00 input 
def test_f1_input_00():
    # f1 is the constant one function, so f1(0) = 1.
    # The oracle computes y XOR f1(x) = 0 XOR 1 = 1, so y flips from 0 to 1.
    # x is unaffected — the X gate in oracle_f1 acts only on the ancilla qubit.
    x_out, y_out = run_oracle(oracle_f1, x_init=0, y_init=0)
    assert x_out == 0 and y_out == 1  # |0⟩|0⟩ → |0⟩|1⟩  (0 XOR 1 = 1)
    print(f"f1 |0⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")

# Tests oracle_f1 with 01 input 
def test_f1_input_01():
    # f1 is the constant one function, so f1(0) = 1.
    # The oracle computes y XOR f1(x) = 1 XOR 1 = 0, so y flips from 1 to 0.
    # This shows the X gate always toggles y — when y starts as 1, it becomes 0.
    x_out, y_out = run_oracle(oracle_f1, x_init=0, y_init=1)
    assert x_out == 0 and y_out == 0  # |0⟩|1⟩ → |0⟩|0⟩  (1 XOR 1 = 0)
    print(f"f1 |0⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f1 with 10 input 
def test_f1_input_10():
    # f1 is the constant one function, so f1(1) = 1.
    # The oracle computes y XOR f1(x) = 0 XOR 1 = 1, so y flips from 0 to 1.
    # Critically, x = 1 here — confirming the flip still occurs regardless of x.
    x_out, y_out = run_oracle(oracle_f1, x_init=1, y_init=0)
    assert x_out == 1 and y_out == 1  # |1⟩|0⟩ → |1⟩|1⟩  (0 XOR 1 = 1)
    print(f"f1 |1⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f1 with 11 input 
def test_f1_input_11():
    # f1 is the constant one function, so f1(1) = 1.
    # The oracle computes y XOR f1(x) = 1 XOR 1 = 0, so y flips from 1 to 0.
    # All four tests together confirm that oracle_f1 always flips y regardless
    # of x — consistent with the unconditional X gate used in its implementation.
    x_out, y_out = run_oracle(oracle_f1, x_init=1, y_init=1)
    assert x_out == 1 and y_out == 0  # |1⟩|1⟩ → |1⟩|0⟩  (1 XOR 1 = 0)
    print(f"f1 |1⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")

#### **oracle_f2 Function Tests:**

In [64]:
# Tests oracle_f2 with 00 input
def test_f2_input_00():
    # f2 is the identity function, so f2(0) = 0.
    # The oracle computes y XOR f2(x) = 0 XOR 0 = 0, so y remains 0.
    # Since x = 0, the CNOT in oracle_f2 does not fire — its control qubit is 0,
    # so the target qubit y is left completely unchanged. Neither qubit changes.
    x_out, y_out = run_oracle(oracle_f2, x_init=0, y_init=0)
    assert x_out == 0 and y_out == 0  # |0⟩|0⟩ → |0⟩|0⟩  (0 XOR 0 = 0)
    print(f"f2 |0⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f2 with 01 input
def test_f2_input_01():
    # f2 is the identity function, so f2(0) = 0.
    # The oracle computes y XOR f2(x) = 1 XOR 0 = 1, so y remains 1.
    # Again x = 0, so the CNOT does not fire — y is left at its initial value of 1.
    # This confirms that when x = 0, oracle_f2 behaves as the identity regardless
    # of y's starting value, consistent with f2(0) = 0 contributing nothing to XOR.
    x_out, y_out = run_oracle(oracle_f2, x_init=0, y_init=1)
    assert x_out == 0 and y_out == 1  # |0⟩|1⟩ → |0⟩|1⟩  (1 XOR 0 = 1)
    print(f"f2 |0⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")

# Tests oracle_f2 with 10 input
def test_f2_input_10():
    # f2 is the identity function, so f2(1) = 1.
    # The oracle computes y XOR f2(x) = 0 XOR 1 = 1, so y flips from 0 to 1.
    # Since x = 1, the CNOT fires — its control qubit is 1, so the target qubit
    # y is flipped. This is the key case that distinguishes oracle_f2 from oracle_f0:
    # the oracle only modifies y when x = 1, directly encoding the identity function.
    x_out, y_out = run_oracle(oracle_f2, x_init=1, y_init=0)
    assert x_out == 1 and y_out == 1  # |1⟩|0⟩ → |1⟩|1⟩  (0 XOR 1 = 1)
    print(f"f2 |1⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")

# Tests oracle_f2 with 11 input
def test_f2_input_11():
    # f2 is the identity function, so f2(1) = 1.
    # The oracle computes y XOR f2(x) = 1 XOR 1 = 0, so y flips from 1 to 0.
    # x = 1 again, so the CNOT fires and toggles y — when y starts as 1 it becomes 0.
    # All four tests together confirm that oracle_f2 flips y if and only if x = 1,
    # which is exactly the behaviour of a CNOT gate and the identity function f2(x) = x.
    x_out, y_out = run_oracle(oracle_f2, x_init=1, y_init=1)
    assert x_out == 1 and y_out == 0  # |1⟩|1⟩ → |1⟩|0⟩  (1 XOR 1 = 0)
    print(f"f2 |1⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")

#### **oracle_f3 Function Tests:**

In [65]:
# Tests oracle_f3 with 00 input
def test_f3_input_00():
    # f3 is the negation function, so f3(0) = NOT 0 = 1.
    # The oracle computes y XOR f3(x) = 0 XOR 1 = 1, so y flips from 0 to 1.
    # Since x = 0, the first X gate in oracle_f3 temporarily flips x to 1,
    # causing the CNOT to fire and flip y. The second X gate then restores x to 0.
    # This is the anti-controlled NOT in action — the flip occurs because x = 0,
    # which is the opposite control condition to oracle_f2's CNOT.
    x_out, y_out = run_oracle(oracle_f3, x_init=0, y_init=0)
    assert x_out == 0 and y_out == 1  # |0⟩|0⟩ → |0⟩|1⟩  (0 XOR NOT 0 = 1)
    print(f"f3 |0⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f3 with 01 input
def test_f3_input_01():
    # f3 is the negation function, so f3(0) = NOT 0 = 1.
    # The oracle computes y XOR f3(x) = 1 XOR 1 = 0, so y flips from 1 to 0.
    # Again x = 0, so the flanking X gates cause the CNOT to fire, toggling y.
    # When y starts as 1 it becomes 0 — confirming the flip always occurs when
    # x = 0, regardless of y's initial value.
    x_out, y_out = run_oracle(oracle_f3, x_init=0, y_init=1)
    assert x_out == 0 and y_out == 0  # |0⟩|1⟩ → |0⟩|0⟩  (1 XOR NOT 0 = 0)
    print(f"f3 |0⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")
 
# Tests oracle_f3 with 10 input
def test_f3_input_10():
    # f3 is the negation function, so f3(0) = NOT 0 = 1.
    # The oracle computes y XOR f3(x) = 1 XOR 1 = 0, so y flips from 1 to 0.
    # Again x = 0, so the flanking X gates cause the CNOT to fire, toggling y.
    # When y starts as 1 it becomes 0 — confirming the flip always occurs when
    # x = 0, regardless of y's initial value.
    x_out, y_out = run_oracle(oracle_f3, x_init=1, y_init=0)
    assert x_out == 1 and y_out == 0  # |1⟩|0⟩ → |1⟩|0⟩  (0 XOR NOT 1 = 0)
    print(f"f3 |1⟩|0⟩ → |{x_out}⟩|{y_out}⟩ ✓")

# Tests oracle_f3 with 11 input
def test_f3_input_11():
    # f3 is the negation function, so f3(1) = NOT 1 = 0.
    # The oracle computes y XOR f3(x) = 0 XOR 0 = 0, so y remains unchanged.
    # Since x = 1, the first X gate flips x to 0, so the CNOT does not fire —
    # its control is now 0 after the flip. The second X gate restores x to 1.
    # This is the key case that distinguishes oracle_f3 from oracle_f1: when
    # x = 1, no flip occurs, because f3(1) = 0 contributes nothing to the XOR.
    x_out, y_out = run_oracle(oracle_f3, x_init=1, y_init=1)
    assert x_out == 1 and y_out == 1  # |1⟩|1⟩ → |1⟩|1⟩  (1 XOR NOT 1 = 1)
    print(f"f3 |1⟩|1⟩ → |{x_out}⟩|{y_out}⟩ ✓")

#### Run the tests from main

In [66]:
# Main function to run the tests for the oracles
if __name__ == "__main__":
    # Run tests for oracle_f0
    print("oracle_f0 tests:")
    test_f0_input_00()
    test_f0_input_01()
    test_f0_input_10()
    test_f0_input_11()
    
    # Run tests for oracle_f1
    print("\noracle_f1 tests:")
    test_f1_input_00()
    test_f1_input_01()
    test_f1_input_10()
    test_f1_input_11()

    # Run tests for oracle_f2
    print("\noracle_f2 tests:")
    test_f2_input_00()
    test_f2_input_01()
    test_f2_input_10()
    test_f2_input_11()
    
    # Run tests for oracle_f3
    print("\noracle_f3 tests:")
    test_f3_input_00()
    test_f3_input_01()
    test_f3_input_10()
    test_f3_input_11()

oracle_f0 tests:
f0 |0⟩|0⟩ → |0⟩|0⟩ ✓
f0 |0⟩|1⟩ → |0⟩|1⟩ ✓
f0 |1⟩|0⟩ → |1⟩|0⟩ ✓
f0 |1⟩|1⟩ → |1⟩|1⟩ ✓

oracle_f1 tests:
f1 |0⟩|0⟩ → |0⟩|1⟩ ✓
f1 |0⟩|1⟩ → |0⟩|0⟩ ✓
f1 |1⟩|0⟩ → |1⟩|1⟩ ✓
f1 |1⟩|1⟩ → |1⟩|0⟩ ✓

oracle_f2 tests:
f2 |0⟩|0⟩ → |0⟩|0⟩ ✓
f2 |0⟩|1⟩ → |0⟩|1⟩ ✓
f2 |1⟩|0⟩ → |1⟩|1⟩ ✓
f2 |1⟩|1⟩ → |1⟩|0⟩ ✓

oracle_f3 tests:
f3 |0⟩|0⟩ → |0⟩|1⟩ ✓
f3 |0⟩|1⟩ → |0⟩|0⟩ ✓
f3 |1⟩|0⟩ → |1⟩|0⟩ ✓
f3 |1⟩|1⟩ → |1⟩|1⟩ ✓


## **Problem 4: Deutsch's Algorithm with Qiskit**

**Brief:** Use Qiskit to design a quantum circuit that solves Deutsch's problem for a function with a single Boolean input. Implement the necessary circuit and demonstrate its use with each of the quantum oracles from Problem 3. Describe how the interference pattern produced by the circuit allows you to determine whether the function is constant or balanced using only one query to the oracle.

### **Introduction**

Problem 4 builds directly on the four oracles implemented in Problem 3. Deutsch's algorithm is the simplest quantum algorithm that demonstrates a provable speedup over any classical approach — it determines whether a Boolean function f: {0,1} → {0,1} is constant (f(0) = f(1)) or balanced (f(0) ≠ f(1)) using only a single query to the oracle. Classically this is impossible in one query, since you must evaluate both f(0) and f(1) separately. The quantum circuit achieves this by placing the input qubit into superposition with a Hadamard gate, querying the oracle once across both inputs simultaneously, then applying a second Hadamard to convert the phase difference introduced by the oracle into a measurable amplitude difference. If the function is constant both paths interfere constructively and the input qubit measures |0⟩. If balanced they interfere destructively and it measures |1⟩ — a single measurement reveals the global property.

### **Function build_deutsch_circuit(oracle_fn):**

The build_deutsch_circuit function constructs the quantum circuit for Deutsch's algorithm for a given oracle. The circuit operates on two qubits: q0 is the input qubit initialised to |0⟩, and q1 is the ancilla qubit initialised to |1⟩ via an X gate. The ancilla must start as |1⟩ so that after its Hadamard gate it enters the |−⟩ = (|0⟩ − |1⟩)/√2 state — this is a prerequisite for phase kickback to work. Without it, the oracle would XOR into |0⟩ and no phase would be imprinted on the input qubit. After initialisation, Hadamard gates are applied to both qubits: q0 enters the uniform superposition |+⟩ = (|0⟩ + |1⟩)/√2, encoding both possible inputs simultaneously, and q1 becomes |−⟩ as described. The oracle is then applied to this two-qubit state. With q1 in |−⟩, the phase kickback mechanism causes the oracle to imprint a phase of (−1)^f(x) onto q0 rather than modifying the ancilla — the transformation |x⟩|−⟩ → (−1)^f(x)|x⟩|−⟩ leaves q1 unchanged and encodes f's output entirely as a phase on q0. Finally, a second Hadamard is applied to q0 alone to interfere the two phase-encoded amplitudes: if f is constant both phases are equal and interfere constructively, collapsing q0 to |0⟩; if f is balanced the phases are opposite and interfere destructively, collapsing q0 to |1⟩. Measuring q0 then directly reveals the answer.

In [67]:
def build_deutsch_circuit(oracle_fn):
    """
    Construct the Deutsch algorithm circuit for a given oracle.
 
    The circuit uses two qubits:
      q0 — input qubit (x): initialised to |0⟩
      q1 — ancilla qubit (y): initialised to |1⟩ via an X gate
 
    Returns the full circuit before measurement so the state can be inspected.
    """

    qr = QuantumRegister(2, 'q')
    cr = ClassicalRegister(1, 'c')
    qc = QuantumCircuit(qr, cr)

    # Initialise ancilla to |1⟩ — required for phase kickback to work
    # Without this, the oracle XORs into |0⟩ and no phase is imprinted
    qc.x(qr[1])
 
    # Apply Hadamard to both qubits
    # q0: |0⟩ → |+⟩ = (|0⟩ + |1⟩)/√2  — superposition over both inputs
    qc.h(qr[0])
    # q1: |1⟩ → |−⟩ = (|0⟩ − |1⟩)/√2  — the phase kickback ancilla state
    qc.h(qr[1])
 
    # visually separate initialisation from oracle
    qc.barrier()  
 
    # Apply the oracle. With q1 in |−⟩, the oracle imprints a phase:
    #   |x⟩|−⟩ → (−1)^f(x) |x⟩|−⟩
    # The ancilla is unchanged; the phase is now on q0.
    oracle_fn(qc, qr[0], qr[1])
 
    # visually separate oracle from interference step
    qc.barrier() 
 
    # Apply Hadamard to q0 only to interfere the two phase-encoded amplitudes
    # Constant f:  phases equal    → constructive interference → |0⟩
    # Balanced f:  phases opposite → destructive interference  → |1⟩
    qc.h(qr[0])
 
    # Measure only q0 — it encodes the constant/balanced answer
    qc.measure(qr[0], cr[0])
 
    return qc

### **Function run_deutsch(oracle_fn):**

The run_deutsch function executes the Deutsch circuit for a given oracle and maps the measurement outcome to a classification. It calls build_deutsch_circuit to construct the circuit, then simulates it using Qiskit's AerSimulator over 1024 shots. Because the result is deterministic — the interference pattern always collapses q0 to either |0⟩ or |1⟩ with certainty — all 1024 shots produce the same bitstring, and the shot count affects only statistical confidence rather than correctness. The dominant measurement outcome is extracted from the counts and mapped directly to a classification: a measured bit of '0' means the function is constant, since constructive interference concentrated all amplitude on |0⟩, and a measured bit of '1' means the function is balanced, since destructive interference redirected all amplitude away from |0⟩. The function returns the classification, the raw counts, and the circuit itself so the result can be inspected and verified by the caller.

In [68]:
def run_deutsch(oracle_fn): 
    """
    Run the Deutsch circuit for the given oracle and return the classification
    ('CONSTANT' or 'BALANCED') along with the raw measurement counts.
    """

    # Create quantum circuit 
    qc = build_deutsch_circuit(oracle_fn)

    # Simulate with 1024 shots. The result is deterministic — all shots produce
    # the same bit — so shot count only affects statistical confidence.
    counts = AerSimulator().run(qc, shots=1024).result().get_counts()
 
    # q0 = '0' → constant, q0 = '1' → balanced
    measured_bit = max(counts, key=counts.get)
    classification = "CONSTANT" if measured_bit == '0' else "BALANCED"
 
    return classification, counts, qc
 

### **Testing**

In [69]:
# ── Single oracle query ───────────────────────────────────────────────────────
# The defining property of Deutsch's algorithm is that it queries the oracle
# exactly once. These tests verify this by inspecting the circuit directly.

def test_circuit_queries_oracle_exactly_once():
    # The Deutsch circuit must contain exactly one barrier-separated oracle
    # section — confirming only a single query is made to determine the result.
    # More than one oracle call would defeat the purpose of the algorithm.
    qc = build_deutsch_circuit(oracle_f0)
    # Count barriers — the circuit has two barriers (before and after oracle),
    # confirming the oracle is applied exactly once between them
    barrier_count = sum(1 for instr in qc.data if instr.operation.name == 'barrier')
    assert barrier_count == 2  # one before oracle, one after — oracle called once
    print(f"Circuit contains {barrier_count} barriers → oracle queried exactly once ✓")

def test_circuit_has_two_hadamards_on_input_qubit():
    # Deutsch's algorithm requires exactly two Hadamard gates on the input qubit
    # q0 — one to create superposition before the oracle, and one to perform
    # interference after. These two gates are what enable a single query to suffice.
    qc = build_deutsch_circuit(oracle_f0)
    h_on_q0 = [instr for instr in qc.data
                if instr.operation.name == 'h'
                and instr.qubits[0] == qc.qregs[0][0]]
    assert len(h_on_q0) == 2  # one for superposition, one for interference
    print(f"q0 has {len(h_on_q0)} Hadamard gates → superposition and interference confirmed ✓")

In [70]:
if __name__ == "__main__":
    test_circuit_queries_oracle_exactly_once()
    test_circuit_has_two_hadamards_on_input_qubit()
    print("\nAll tests passed!")

Circuit contains 2 barriers → oracle queried exactly once ✓
q0 has 2 Hadamard gates → superposition and interference confirmed ✓

All tests passed!


In [71]:
if __name__ == "__main__":
 
    # Each entry is (display name, oracle function, expected classification).
    # The expected value is used to verify the algorithm gives the correct answer.
    oracles = [
        ("f0: f(x) = 0",   oracle_f0, "CONSTANT"),
        ("f1: f(x) = 1",   oracle_f1, "CONSTANT"),
        ("f2: f(x) = x",   oracle_f2, "BALANCED"),
        ("f3: f(x) = ¬x",  oracle_f3, "BALANCED"),
    ]
 
    print("=" * 60)
    print("  Problem 4: Deutsch's Algorithm")
    print("=" * 60)
 
    # Run the Deutsch circuit once per oracle and print the outcome.
    # Each oracle only needs a single query to classify — no repetition needed.
    for name, oracle_fn, expected in oracles:
        classification, counts, qc = run_deutsch(oracle_fn)

        # Compare against the known answer to confirm the algorithm is correct
        correct = classification == expected
 
        print(f"\n  Oracle : {name}")
        print(f"  Circuit:\n{qc.draw('text', fold=-1)}")
        print(f"  Counts : {counts}")
        print(f"  Result : {classification} ({'✓ correct' if correct else '✗ wrong'})")
 
    print("\n" + "=" * 60)
    print("  How the interference works")
    print("=" * 60)
    print("""
  After the Hadamard gates, the two-qubit state is:
 
    (|0⟩ + |1⟩)/√2  ⊗  (|0⟩ − |1⟩)/√2
 
  The oracle maps |x⟩|−⟩ → (−1)^f(x)|x⟩|−⟩, so q0 becomes:
 
    (−1)^f(0)|0⟩ + (−1)^f(1)|1⟩  (up to normalisation)
 
  The final Hadamard on q0 then gives:
 
    Constant f — f(0)=f(1): phases equal, interference is constructive
      → amplitude concentrates on |0⟩ → measures 0
 
    Balanced f — f(0)≠f(1): phases opposite, interference is destructive
      → amplitude concentrates on |1⟩ → measures 1
 
  One oracle call encodes both f(0) and f(1) simultaneously through
  superposition, and interference converts the phase relationship into
  a directly measurable bit.
""")

  Problem 4: Deutsch's Algorithm

  Oracle : f0: f(x) = 0
  Circuit:
     ┌───┐      ░  ░ ┌───┐┌─┐
q_0: ┤ H ├──────░──░─┤ H ├┤M├
     ├───┤┌───┐ ░  ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░──░───────╫─
     └───┘└───┘ ░  ░       ║ 
c: 1/══════════════════════╩═
                           0 
  Counts : {'0': 1024}
  Result : CONSTANT (✓ correct)

  Oracle : f1: f(x) = 1
  Circuit:
     ┌───┐      ░       ░ ┌───┐┌─┐
q_0: ┤ H ├──────░───────░─┤ H ├┤M├
     ├───┤┌───┐ ░ ┌───┐ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤ X ├─░───────╫─
     └───┘└───┘ ░ └───┘ ░       ║ 
c: 1/═══════════════════════════╩═
                                0 
  Counts : {'0': 1024}
  Result : CONSTANT (✓ correct)

  Oracle : f2: f(x) = x
  Circuit:
     ┌───┐      ░       ░ ┌───┐┌─┐
q_0: ┤ H ├──────░───■───░─┤ H ├┤M├
     ├───┤┌───┐ ░ ┌─┴─┐ ░ └───┘└╥┘
q_1: ┤ X ├┤ H ├─░─┤ X ├─░───────╫─
     └───┘└───┘ ░ └───┘ ░       ║ 
c: 1/═══════════════════════════╩═
                                0 
  Counts : {'1': 1024}
  Result : BALANCED (✓ cor

### **Problem 4 Testing**

## **Problem 5: Scaling to the Deutsch–Jozsa Algorithm**

**Brief:** The Deutsch–Jozsa algorithm generalizes Deutsch's approach to functions with several input bits. Use Qiskit to create a quantum circuit that can handle the four-bit functions generated in Problem 1. Explain how the classical function is encoded as a quantum oracle, and demonstrate the use of your circuit on both of the constant functions and any two balanced functions of your choosing. Show that the circuit correctly identifies the type of each function.

**Introduction:** Problem 5 generalises Deutsch's algorithm from a single input bit to multiple input bits. The Deutsch–Jozsa algorithm determines whether a function f: {0,1}ⁿ → {0,1} is constant or balanced in a single oracle query, regardless of how many input bits n there are. Classically, in the worst case, you would need 2ⁿ⁻¹ + 1 evaluations to be certain. Here n = 4, as the functions were generated in Problem 1. The circuit structure is a direct generalisation of Deutsch's: n input qubits are placed into superposition, the oracle is queried once, and a final layer of Hadamard gates produces an interference pattern that is all |0⟩s if and only if the function is constant. The key challenge in this problem is encoding the classical four-bit function as a quantum oracle — a phase oracle that maps |x⟩ → (−1)^f(x)|x⟩ via the same phase kickback mechanism used in Problem 3.

### **build_oracle(classical_fn) function:**

The build_oracle function converts a classical 4-input Boolean function — as generated by Problem 1 — into a quantum oracle circuit. The oracle iterates over all 16 possible inputs and for each input where f(x) = True, constructs a gate sequence that flips the ancilla qubit q4 if and only if the input register holds exactly that input. This is achieved using the same X-MCX-X pattern introduced in oracle_f3 in Problem 3, scaled up to 4 control qubits. For each target input, X gates are applied to any qubits that are False in that input, temporarily flipping them to True so that the multi-controlled X gate (MCX) fires on the correct computational basis state. The MCX then flips the ancilla, encoding y ⊕ f(x) on q4. Finally, the X gates are reapplied to restore the input qubits to their original state, preserving the unitary requirement that the input register is left unchanged. Inputs where f(x) = False require no gates at all, since y ⊕ 0 = y is the identity.

In [72]:
"""
Problem 5: Scaling to the Deutsch-Jozsa Algorithm (4-bit)
==========================================================
Generalises Deutsch's algorithm (Problem 4) to f: {0,1}^4 → {0,1}.

Builds directly on:
  - Problem 1: random_constant_balanced() generates the functions under test
  - Problem 2: determine_constant_balanced() provides the classical answer to
               verify against and illustrate the classical vs quantum cost
  - Problem 3: same phase kickback oracle construction, scaled to 4 input qubits
  - Problem 4: same H-oracle-H circuit structure, scaled to 4 input qubits

The circuit measures all zeros on q0-q3 if and only if the function is constant.
"""

# ──────────────────────────────────────────────────────────────────────────────
# Step 1 — Encode the classical function as a quantum oracle
# ──────────────────────────────────────────────────────────────────────────────

# All 16 possible 4-bit inputs — used to iterate over the function's truth table
ALL_INPUTS = list(product([False, True], repeat=4))

def build_oracle(classical_fn) -> QuantumCircuit:
    """
    Convert a classical 4-input Boolean function (from Problem 1) into a
    quantum oracle using the same phase kickback pattern as Problem 3.

    For each input x where f(x) = True, a multi-controlled X gate (MCX) is
    applied to the ancilla qubit, conditioned on the input register holding x.
    X gates flanking the MCX invert any False-bits in x so the MCX fires on
    exactly the right computational basis state — the same X-CNOT-X pattern
    used in oracle_f3 in Problem 3, generalised to 4 control qubits.
    """
    n = 4
    qr = QuantumRegister(n + 1, 'q')  # q0-q3: input qubits, q4: ancilla
    qc = QuantumCircuit(qr)

    for x in ALL_INPUTS:
        if classical_fn(*x):  # only act on inputs where f(x) = True
            # Flip qubits that are False in this input pattern so the MCX
            # fires when the register holds exactly x
            for i, bit in enumerate(x):
                if not bit:
                    qc.x(qr[i])

            # MCX flips ancilla q4 iff q0-q3 match x — encodes y XOR f(x)
            qc.mcx(list(range(n)), n)

            # Restore the input qubits to their original state
            for i, bit in enumerate(x):
                if not bit:
                    qc.x(qr[i])

    return qc

### **build_dj_circuit(classical_fn) function:**

The build_dj_circuit function constructs the full Deutsch-Jozsa circuit for a given 4-input Boolean function — a direct generalisation of the Deutsch circuit from Problem 4, scaled from 1 input qubit to 4. The circuit follows the same three-stage structure: first, all qubits are initialised and Hadamarded into superposition; second, the oracle is applied; third, a final layer of Hadamard gates produces an interference pattern that encodes the answer. The ancilla qubit q4 is initialised to |1⟩ before the Hadamard layer, placing it in the |−⟩ state required for phase kickback — identical to the setup in Problem 4. After the oracle imprints a phase of (−1)^f(x) on each input basis state, the Hadamard gates on q0–q3 interfere those phases: for a constant function all phases are equal and interfere constructively, concentrating the entire probability amplitude on the all-zeros state |0000⟩; for a balanced function the phases cancel at |0000⟩ and the measurement always produces a non-zero bitstring. Only the four input qubits are measured — the ancilla is discarded — and a result of 0000 identifies the function as constant while any other result identifies it as balanced.

In [73]:
# ──────────────────────────────────────────────────────────────────────────────
# Step 2 — Build the Deutsch-Jozsa circuit
# ──────────────────────────────────────────────────────────────────────────────

def build_dj_circuit(classical_fn) -> QuantumCircuit:
    """
    Build the full Deutsch-Jozsa circuit — a direct generalisation of the
    Deutsch circuit from Problem 4, scaled from 1 to 4 input qubits.

    Problem 4 circuit:    |0⟩ ──H──[oracle]──H── measure
    This circuit:    |0⟩⊗4 ──H⊗4──[oracle]──H⊗4── measure (on q0-q3 only)

    The ancilla qubit q4 is initialised to |1⟩ and Hadamarded into |−⟩ for
    phase kickback, exactly as in Problem 4.
    """
    n = 4
    qr = QuantumRegister(n + 1, 'q')
    cr = ClassicalRegister(n, 'c')
    qc = QuantumCircuit(qr, cr)

    # Initialise ancilla to |1⟩ — same as Problem 4, needed for phase kickback
    qc.x(qr[n])

    # Hadamard on all qubits: input qubits enter uniform superposition,
    # ancilla becomes |−⟩ = (|0⟩ − |1⟩)/√2
    qc.h(qr[:])

    qc.barrier()

    # Apply oracle — imprints (−1)^f(x) phase onto the input register,
    # using the same phase kickback mechanism as Problem 3
    oracle = build_oracle(classical_fn)
    qc.compose(oracle, inplace=True)

    qc.barrier()

    # Hadamard on input qubits only — interferes the phase-encoded amplitudes.
    # Constant f → all phases equal → constructive interference → |0000⟩
    # Balanced f → phases cancel at |0000⟩ → non-zero result
    qc.h(qr[:n])

    # Measure input qubits only — ancilla is discarded
    qc.measure(qr[:n], cr)

    return qc


### **Runner function - run_dj(classical_fn):** 

The run_dj function executes the Deutsch-Jozsa circuit for a given 4-input Boolean function and returns the classification. It calls build_dj_circuit to construct the circuit, simulates it over 1024 shots, and maps the dominant measurement outcome to either "constant" or "balanced". The result is deterministic — all 1024 shots always agree — so the classification is read directly from whichever bitstring dominates the counts. A measurement of '0000' on q0–q3 indicates a constant function, since constructive interference concentrated all amplitude on the all-zeros state. Any other outcome indicates a balanced function, since destructive interference prevented any amplitude from reaching '0000'.

In [74]:
def run_dj(classical_fn) -> tuple[str, dict, QuantumCircuit]:
    """
    Run the Deutsch-Jozsa circuit for a function generated by Problem 1.
    Returns the quantum classification, raw counts, and the circuit.
 
    Measurement of '0000' → constant (constructive interference)
    Any other outcome    → balanced (destructive interference at |0000⟩)
    """
    
    qc = build_dj_circuit(classical_fn)
 
    # Result is deterministic — all 1024 shots produce the same bitstring
    counts = AerSimulator().run(qc, shots=1024).result().get_counts()
    outcome = max(counts, key=counts.get)
 
    classification = "constant" if all(b == '0' for b in outcome) else "balanced"
 
    return classification, counts, qc

### **Testing**

In [75]:
# ──────────────────────────────────────────────────────────────────────────────
# Step 4 — Demonstrate on two constant and two balanced functions
# ──────────────────────────────────────────────────────────────────────────────
 
if __name__ == "__main__":
 
    print("=" * 65)
    print("  Problem 5: Deutsch-Jozsa Algorithm (4-bit)")
    print("=" * 65)
 
    # Use Problem 1 to generate two constant and two balanced functions,
    # re-sampling until we have the right mix for demonstration
    constant_fns = []
    balanced_fns = []
 
    while len(constant_fns) < 2 or len(balanced_fns) < 2:
        f = random_constant_balanced()
        # Use Problem 2 to label the function so we can sort it
        label = determine_constant_balanced(f)
        if label == "constant" and len(constant_fns) < 2:
            constant_fns.append(f)
        elif label == "balanced" and len(balanced_fns) < 2:
            balanced_fns.append(f)
 
    functions_to_test = (
        [(f, "constant") for f in constant_fns] +
        [(f, "balanced") for f in balanced_fns]
    )
 
    for i, (fn, expected) in enumerate(functions_to_test, 1):
 
        # Classical answer from Problem 2 — requires up to 16 calls to f
        classical_result = determine_constant_balanced(fn)
 
        # Quantum answer from Problem 5 — requires exactly 1 oracle query
        quantum_result, counts, qc = run_dj(fn)
 
        # Both should agree — if not, something is wrong with the oracle encoding
        agree = classical_result == quantum_result
        correct = quantum_result == expected
 
        print(f"\n  Function {i} (expected: {expected})")
        print(f"  Classical result (Problem 2) : {classical_result}  [up to 16 calls]")
        print(f"  Quantum result   (Problem 5) : {quantum_result}  [1 oracle query]")
        print(f"  Measurement counts           : {counts}")
        print(f"  Correct? {'✓ yes' if correct else '✗ no'}  |  Classical and quantum agree? {'✓ yes' if agree else '✗ no'}")
 
    print("\n" + "=" * 65)
    print("  Classical vs Quantum cost")
    print("=" * 65)
    print("""
  Problem 2 (classical): must evaluate f on every input to be certain.
    Worst case: 2^(n-1) + 1 = 9 evaluations for n=4.
    This implementation always calls f all 16 times.
 
  Problem 5 (quantum): one oracle query regardless of n.
    The uniform superposition feeds all 16 inputs simultaneously,
    and interference on the output reveals the global property
    without ever evaluating individual inputs.
 
  Speedup: exponential — O(1) queries vs O(2^n) classical.
""")

  Problem 5: Deutsch-Jozsa Algorithm (4-bit)

  Function 1 (expected: constant)
  Classical result (Problem 2) : constant  [up to 16 calls]
  Quantum result   (Problem 5) : constant  [1 oracle query]
  Measurement counts           : {'0000': 1024}
  Correct? ✓ yes  |  Classical and quantum agree? ✓ yes

  Function 2 (expected: constant)
  Classical result (Problem 2) : constant  [up to 16 calls]
  Quantum result   (Problem 5) : constant  [1 oracle query]
  Measurement counts           : {'0000': 1024}
  Correct? ✓ yes  |  Classical and quantum agree? ✓ yes

  Function 3 (expected: balanced)
  Classical result (Problem 2) : balanced  [up to 16 calls]
  Quantum result   (Problem 5) : balanced  [1 oracle query]
  Measurement counts           : {'0010': 65, '1101': 72, '0110': 50, '1000': 74, '1001': 67, '1100': 62, '0111': 63, '0011': 55, '1110': 246, '0101': 270}
  Correct? ✓ yes  |  Classical and quantum agree? ✓ yes

  Function 4 (expected: balanced)
  Classical result (Problem 2) : ba